# Discovery + Silver: `university.semesters`

Tabla chica (8 filas). Mismo patron, mas breve.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.university__semesters", engine)
df

,semester_id,code,year,half,start_date,end_date,_source_file,_ingested_at,_dag_run_id
0,SEM-001,2022-1,2022,1,2022-03-01,2022-07-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
1,SEM-002,2022-2,2022,2,2022-08-01,2022-12-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
2,SEM-003,2023-1,2023,1,2023-03-01,2023-07-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
3,SEM-004,2023-2,2023,2,2023-08-01,2023-12-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
4,SEM-005,2024-1,2024,1,2024-03-01,2024-07-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
5,SEM-006,2024-2,2024,2,2024-08-01,2024-12-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
6,SEM-007,2025-1,2025,1,2025-03-01,2025-07-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00
7,SEM-008,2025-2,2025,2,2025-08-01,2025-12-15,university/semesters.csv,2026-07-15 22:29:47.749144,manual__2026-07-15T22:29:45+00:00


## 1. Chequeos (nulos, duplicados, rangos, fechas)

In [2]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("semester_id duplicados:", df["semester_id"].duplicated().sum())
print("Valores de half:", sorted(df["half"].unique()))

start = pd.to_datetime(df["start_date"])
end = pd.to_datetime(df["end_date"])
print("start_date >= end_date (invertido):", (start >= end).sum())

Nulos por columna:
semester_id     0
code            0
year            0
half            0
start_date      0
end_date        0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

semester_id duplicados: 0
Valores de half: ['1', '2']
start_date >= end_date (invertido): 0


## 2. Conclusion

Tabla limpia. Solo tipado: `year`/`half` a entero, fechas a `date`, `code` sin espacios.

## 3. Limpieza con pandas

In [3]:
df_silver = df[["semester_id", "code", "year", "half", "start_date", "end_date"]].copy()

df_silver["code"] = df_silver["code"].str.strip()
df_silver["year"] = pd.to_numeric(df_silver["year"], errors="raise").astype(int)
df_silver["half"] = pd.to_numeric(df_silver["half"], errors="raise").astype(int)
df_silver["start_date"] = pd.to_datetime(df_silver["start_date"]).dt.date
df_silver["end_date"] = pd.to_datetime(df_silver["end_date"]).dt.date

df_silver

,semester_id,code,year,half,start_date,end_date
0,SEM-001,2022-1,2022,1,2022-03-01,2022-07-15
1,SEM-002,2022-2,2022,2,2022-08-01,2022-12-15
2,SEM-003,2023-1,2023,1,2023-03-01,2023-07-15
3,SEM-004,2023-2,2023,2,2023-08-01,2023-12-15
4,SEM-005,2024-1,2024,1,2024-03-01,2024-07-15
5,SEM-006,2024-2,2024,2,2024-08-01,2024-12-15
6,SEM-007,2025-1,2025,1,2025-03-01,2025-07-15
7,SEM-008,2025-2,2025,2,2025-08-01,2025-12-15


## 4. Validar y escribir

In [4]:
assert len(df_silver) == len(df)
assert df_silver.isna().sum().sum() == 0
assert df_silver["semester_id"].is_unique

df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()
df_silver.to_sql(
    "university__semesters",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.university__semesters")

Escrito en silver.university__semesters


## 5. Verificar

In [5]:
pd.read_sql("SELECT * FROM silver.university__semesters ORDER BY year, half", engine)

,semester_id,code,year,half,start_date,end_date,_silver_loaded_at
0,SEM-001,2022-1,2022,1,2022-03-01,2022-07-15,2026-07-16 19:03:22.928176+00:00
1,SEM-002,2022-2,2022,2,2022-08-01,2022-12-15,2026-07-16 19:03:22.928176+00:00
2,SEM-003,2023-1,2023,1,2023-03-01,2023-07-15,2026-07-16 19:03:22.928176+00:00
3,SEM-004,2023-2,2023,2,2023-08-01,2023-12-15,2026-07-16 19:03:22.928176+00:00
4,SEM-005,2024-1,2024,1,2024-03-01,2024-07-15,2026-07-16 19:03:22.928176+00:00
5,SEM-006,2024-2,2024,2,2024-08-01,2024-12-15,2026-07-16 19:03:22.928176+00:00
6,SEM-007,2025-1,2025,1,2025-03-01,2025-07-15,2026-07-16 19:03:22.928176+00:00
7,SEM-008,2025-2,2025,2,2025-08-01,2025-12-15,2026-07-16 19:03:22.928176+00:00
